# AgriSpectra-Q — Hackathon Colab Notebook

**Planet Tanager STAC discovery + reproducible hyperspectral workflow + AgriSpectra-Q EnMAP application integration**

This notebook is the single reproducible entry point prepared for the Hackathon requirements. It keeps the two scientific tracks separate:

- **EnMAP Live Matrix:** the existing AgriSpectra-Q application path, its sample outputs, maps, zones, and inspection-budget baseline.
- **Planet Tanager:** the course-based STAC scene-finder and HDF5 workflow from Notebooks 16 and 12. A real Tanager HDF5 scene is required before publishing Tanager experiment results; this notebook never fabricates them.

> AgriSpectra-Q produces relative hyperspectral evidence for agricultural inspection prioritization. Its outputs are not confirmed crop diagnoses. Independent field observations and agronomic review are required before intervention.

## Requirement mapping

| Hackathon requirement | Notebook section | Status in this run |
|---|---|---|
| Research question and area | Sections 1–2 | Implemented |
| Planet Tanager STAC access, scene IDs, dates, cloud cover | Section 3 | Implemented from the public course index |
| Bad bands, masks, subset, preprocessing | Sections 4–5 | Implemented as a Tanager-ready path; real HDF5 is conditional |
| Analysis and baseline | Sections 6–7 | EnMAP anomaly pipeline + random inspection baseline |
| Results, plots, units, legends | Sections 7–9 | Implemented on committed project outputs |
| AI log, limitations, next steps | Sections 10–11 | Implemented |
| Clean restart/run-all | Final cell | Designed and locally executed where data are available |

In [ ]:
# Colab/local setup. No secrets or API keys are required by this notebook.
import os, sys, subprocess, pathlib, json, hashlib, shutil, warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PROJECT_REPO = "https://github.com/FreelancerDevBaggash/AgriSpectra-Q-.git"
COURSE_REPO = "https://github.com/Tnecniv-Teikram/NSA-hyperspectral-course-2026.git"
if IN_COLAB:
    ROOT = Path("/content/AgriSpectra-Q-")
    COURSE = Path("/content/hyperspectral-course")
    if not ROOT.exists(): subprocess.run(["git","clone","--depth","1","-q",PROJECT_REPO,str(ROOT)],check=True)
    if not COURSE.exists(): subprocess.run(["git","clone","--depth","1","-q",COURSE_REPO,str(COURSE)],check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","numpy","pandas","matplotlib","rasterio","geopandas","shapely","h5py","scipy","scikit-learn","xarray","rioxarray","pystac-client","requests"],check=True)
else:
    ROOT = Path.cwd()
    if ROOT.name == "notebooks": ROOT = ROOT.parent
    COURSE = ROOT.parent / "course"
    if not (COURSE / "hsi_utils.py").exists():
        COURSE = Path("/home/ubuntu/work/course")

sys.path.insert(0, str(COURSE))
os.chdir(ROOT)
print("ROOT:", ROOT)
print("COURSE:", COURSE)
print("Python:", sys.version.split()[0])

In [ ]:
# Imports and reproducibility settings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
np.random.seed(42)
SEED = 42
OUT = ROOT / "colab_outputs"
OUT.mkdir(exist_ok=True)
INTEGRATION = ROOT / "deliverables" / "agrispectraq_integration_package"
assert INTEGRATION.exists(), "Integration package is missing. Commit deliverables/agrispectraq_integration_package/ with this notebook."
print("Integration package:", INTEGRATION)
print("Attribution: Planet Labs PBC, Tanager Core Imagery, CC-BY-4.0 (when Tanager data are used).")

## 1. Research question and application connection

**Application question:** Given a finite field-inspection budget, which spatial locations should agricultural teams inspect earlier because their hyperspectral signal is unusual relative to the scene?

The output is a relative prioritization signal. It is not a diagnosis of disease, pests, nutrient deficiency, or crop stress. The current application uses EnMAP Live Matrix outputs for the demonstrated decision-support path, while Tanager is evaluated as a separate scientific experiment and data-access track.

## 2. Existing AgriSpectra-Q application artifacts

The repository already contains a reproducible sample demonstration and precomputed full-scene outputs. This section reads those artifacts, rather than claiming that the committed sample crops are full-scene validation.

In [ ]:
# Locate and inspect the primary committed sample-demo run
sample_root = ROOT / "results" / "sample_demo"
runs = sorted(sample_root.glob("*/scene_01_sudan_sample"))
assert runs, f"No sample-demo run found under {sample_root}"
SAMPLE_RUN = runs[0]
print("Primary sample run:", SAMPLE_RUN)
required = ["risk_map.tif","priority_map.tif","scene_statistics.json","manifest.json","zones.csv","zones.geojson","inspection_budget.csv"]
for name in required:
    print(f"{'OK' if (SAMPLE_RUN/name).exists() else 'MISSING':7s} {name}")
stats = json.loads((SAMPLE_RUN/"scene_statistics.json").read_text())
manifest = json.loads((SAMPLE_RUN/"manifest.json").read_text())
zones = pd.read_csv(SAMPLE_RUN/"zones.csv")
budget = pd.read_csv(SAMPLE_RUN/"inspection_budget.csv")
print("\nScene:", stats.get("scene"), "location:", stats.get("location"), "CRS:", stats.get("crs"))
print("Dimensions:", stats.get("dimensions"), "valid pixels:", stats.get("valid_pixels"))
print("Zones:", len(zones), "columns:", list(zones.columns))
display(budget.head())

## 3. Planet Tanager scene discovery from Notebook 16

Notebook 16 uses the course helper `hsi_utils.load_catalog_index()` to search the static STAC catalogue. The filters below expose collection, location, date, cloud cover, and viewing angle. A thumbnail or cloud percentage is evidence to inspect, not an automatic scientific truth.

In [ ]:
import hsi_utils as hu
import geopandas as gpd
from shapely.geometry import box, Point

index = hu.load_catalog_index(refresh=False)
print(f"{len(index)} catalogue rows; {index['id'].nunique()} unique scene IDs")
print("Collections:", sorted(index["collection"].dropna().unique().tolist()))
cols_per_scene = index.groupby("id")["collection"].apply(lambda s: ", ".join(sorted(set(s))))
scenes = (index.drop_duplicates("id").drop(columns="collection")
          .merge(cols_per_scene.rename("collections"), left_on="id", right_index=True)
          .sort_values("datetime").reset_index(drop=True))
scenes["country"] = scenes["location"].astype(str).str.split(",").str[-1].str.strip()
scenes["month"] = scenes["datetime"].dt.strftime("%Y-%m")
centroids = scenes.to_crs(6933).centroid.to_crs(4326)
scenes["lon"], scenes["lat"] = centroids.x, centroids.y

def search_scenes(df, bbox=None, point=None, buffer_km=0, start=None, end=None, max_cloud=None, max_off_nadir=None, collections=None):
    out = df
    if bbox is not None: out = out[out.intersects(box(*bbox))]
    if point is not None:
        aoi = gpd.GeoSeries([Point(point)], crs=4326).to_crs(6933).buffer(buffer_km*1000+1).to_crs(4326).iloc[0]
        out = out[out.intersects(aoi)]
    if start is not None: out = out[out["datetime"] >= pd.Timestamp(start, tz="UTC")]
    if end is not None: out = out[out["datetime"] <= pd.Timestamp(end, tz="UTC")]
    if max_cloud is not None: out = out[out["cloud_percent"] <= max_cloud]
    if max_off_nadir is not None: out = out[out["off_nadir"] <= max_off_nadir]
    if collections is not None: out = out[out["collections"].apply(lambda s: any(c in s.split(", ") for c in collections))]
    return out.copy()

ARAB_BBOX = (-17.5, 12.0, 60.0, 38.0)
arab = search_scenes(scenes, bbox=ARAB_BBOX, collections=["agriculture","natural-lands","urban","coastal-water-bodies"])
show = [c for c in ["id","collections","datetime","location","cloud_percent","off_nadir","thumbnail"] if c in arab.columns]
print("Candidate scenes in the study box:", len(arab))
display(arab[show].head(20))
arab.to_file(OUT/"tanager_candidates.geojson", driver="GeoJSON")

In [ ]:
# Catalogue overview plot: required data inspection before choosing a scene
fig, ax = plt.subplots(1,2,figsize=(13,4))
index["collection"].value_counts().sort_values().plot.barh(ax=ax[0], color="#2f6f9f")
ax[0].set_title("Tanager catalogue rows by collection"); ax[0].set_xlabel("Rows")
ax[1].hist(scenes["cloud_percent"].dropna(), bins=np.arange(0,105,5), color="#777", edgecolor="white")
ax[1].set_title("Scene metadata cloud cover"); ax[1].set_xlabel("Cloud cover (%)"); ax[1].set_ylabel("Scenes")
plt.tight_layout(); plt.show()

## 4. Scene selection decision

For a production run, select one or more candidate `Scene IDs` after checking the footprint, acquisition date, cloud percentage, off-nadir angle, and thumbnail. The notebook records IDs rather than downloading raw scenes into Drive. The following cell creates a reproducible selection table; it does not download a ~1 GB scene by default.

In [ ]:
# Explicit, reviewable selection policy. Change only after inspecting the table/thumbnails.
SELECTION_POLICY = {"max_cloud_percent": 60, "max_off_nadir_deg": 25, "require_collection": "agriculture"}
selected = search_scenes(scenes, bbox=ARAB_BBOX, max_cloud=SELECTION_POLICY["max_cloud_percent"], max_off_nadir=SELECTION_POLICY["max_off_nadir_deg"], collections=[SELECTION_POLICY["require_collection"]])
selected = selected.sort_values(["cloud_percent","off_nadir","datetime"], na_position="last")
SCENE_IDS = selected["id"].head(5).tolist()
print("Selected candidate Scene IDs (review before download):")
for sid in SCENE_IDS: print(" -", sid)
selected[[c for c in ["id","datetime","location","cloud_percent","off_nadir","collections"] if c in selected.columns]].to_csv(OUT/"selected_tanager_scene_ids.csv", index=False)
print("Saved:", OUT/"selected_tanager_scene_ids.csv")

## 5. Tanager HDF5 input, bad bands, masks, and subset
The supplied Planet Tanager HDF5 is approximately 1 GB, so the notebook never loads the full spectral cube into memory. It records the Scene ID, SHA-256, metadata, dimensions, valid bands, and valid pixels. The raw HDF5 remains in the ephemeral Colab runtime and is not written to Drive or GitHub.


In [ ]:
# Download the supplied public Drive HDF5 to the ephemeral Colab runtime.
# It is not persisted to Google Drive or committed to GitHub.
TANAGER_FILE_ID = "1s3To4_DPWmBJ2DftkpCTfnWe2HN6EsyP"
TANAGER_FILENAME = "20250501_143138_87_4001_ortho_sr_hdf5.h5"
TANAGER_HDF5_PATH = Path("/content") / TANAGER_FILENAME if IN_COLAB else Path("/home/ubuntu/work/tanager") / TANAGER_FILENAME
if IN_COLAB and not TANAGER_HDF5_PATH.exists():
    url = f"https://drive.usercontent.google.com/download?id={TANAGER_FILE_ID}&export=download&confirm=t"
    subprocess.run(["curl","-L","--fail","--retry","2","-o",str(TANAGER_HDF5_PATH),url], check=True)
print("Tanager file:", TANAGER_HDF5_PATH)
assert TANAGER_HDF5_PATH.exists(), "Supply the approved HDF5 at the documented path before running."
digest = hashlib.sha256(TANAGER_HDF5_PATH.read_bytes()).hexdigest()
print("size_MB:", round(TANAGER_HDF5_PATH.stat().st_size/1e6,2), "sha256:", digest)

In [ ]:
# Real Tanager structural inspection and memory-safe subset QA.
scene = hu.TanagerScene(TANAGER_HDF5_PATH)
print(scene)
print(scene.summary())
print("Field names:", scene.field_names)
print("Good bands:", int(scene.good_bands.sum()), "/", len(scene.good_bands))
window = (100, min(356, scene.shape[1]), 100, min(356, scene.shape[2]))
masks = scene.masks(window)
valid = scene.valid_mask(window, cloud=True)
print("Subset:", window, "valid_fraction:", round(float(valid.mean()),4))
print("Mask fractions:", {k: round(float(v.mean()),4) for k,v in masks.items()})
TANAGER_STATUS = "PASS — real HDF5 inspected and subset QA completed"

## 6. One integrated Tanager workflow: Tile-by-Tile passes
The course material is integrated here as one reproducible path: Notebook 16 provides STAC discovery and Scene IDs; Notebook 12 provides lazy HDF5 reading, good-band filtering, masks, window subsets, and provenance. The final Tanager engine is explicitly Tile-by-Tile:

1. **Pass 1A:** read small windows and estimate streaming mean/std plus a bounded spectral reservoir for median/MAD.
2. **Pass 1B:** read each tile, compute TRUST, and write a temporary windowed GeoTIFF to obtain scene-relative thresholds.
3. **Pass 2:** read each tile, compute baseline/TRUST-derived products, and write GeoTIFF windows directly. Only the 2-D class mask is materialized afterward for connected components.

This design is intended for the approximately 1 GB scene and preserves the same CRS, transform, masks, provenance, and relative-prioritization interpretation.


In [ ]:
# Real-scene Tile-by-Tile run. The raw HDF5 is read by windows; it is never copied to Drive/GitHub.
tanager_out = ROOT / "deliverables" / "tanager_trust_final"
TILE_SIZE = 128
STATISTICS_STRIDE = 2
subprocess.run([
    sys.executable, str(ROOT/"scripts"/"run_tanager_experiment.py"),
    "--source", str(TANAGER_HDF5_PATH), "--output", str(tanager_out),
    "--tile-size", str(TILE_SIZE), "--stride", str(STATISTICS_STRIDE)
], check=True)
repair = json.loads((tanager_out/"repair_metrics.json").read_text())
meta = json.loads((tanager_out/"TANAGER_HDF5_METADATA.json").read_text())
print(json.dumps({"scene":meta["scene_id"], "good_bands":meta["good_band_count"],
                  "valid_fraction":meta["valid_fraction"], "zones":meta["zones_count"],
                  "tile_size":meta["tile_size"], "statistics_stride":meta["statistics_stride"],
                  "method":meta["method"], "repair":repair}, indent=2))
TANAGER_STATUS = "PASS — supplied real HDF5 processed Tile-by-Tile; independent/cross-sensor validation BLOCKED"


## 7. Analysis methodology and baseline

The existing AgriSpectra-Q application computes a scene-relative spectral anomaly score and ranks high-priority connected zones. The simple operational baseline is random inspection: at an inspection budget of `b`, expected recall is approximately `b`. This baseline is not a disease ground truth.

In [ ]:
# Existing application result: inspection-budget curve and random baseline
budget = pd.read_csv(SAMPLE_RUN/"inspection_budget.csv")
budget["random_baseline_recall"] = budget["budget_fraction"]
display(budget[[c for c in ["budget_fraction","coverage_percentage","positive_recall","random_baseline_recall"] if c in budget.columns]])
fig, ax = plt.subplots(figsize=(7,4.5))
ax.plot(budget["coverage_percentage"], budget["positive_recall"]*100, "o-", label="AgriSpectra-Q priority ranking")
ax.plot(budget["coverage_percentage"], budget["random_baseline_recall"]*100, "--", color="gray", label="Random baseline")
ax.set(xlabel="Inspection budget (% of valid pixels)", ylabel="Recall of high-priority proxy (%)", title="Inspection-budget comparison")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 8. Advanced spectral atlas integration
The atlas is generated from the same run artifacts, not from a separate or hand-edited visualization. For the application sample it uses the package's original four-panel contract. For Planet Tanager it uses the same four-panel generator with Tanager filenames: TRUST anomaly, relative priority classes, threshold ambiguity proxy, and P95 footprint. Ambiguity is not calibrated measurement uncertainty. NoData remains unavailable.


In [ ]:
# Generate and validate the package atlas for the existing AgriSpectra-Q sample track.
import subprocess, sys
atlas_tool = INTEGRATION/"tools"/"generate_spectral_atlas.py"
validator = INTEGRATION/"tools"/"validate_spectral_atlas.py"
atlas_path = OUT/"scene_01_sudan_sample_advanced_atlas.png"
subprocess.run([sys.executable, str(atlas_tool), str(SAMPLE_RUN), "--output", str(atlas_path), "--title", "AgriSpectra-Q sample demo"], check=True)
validation = subprocess.run([sys.executable, str(validator), str(SAMPLE_RUN)], capture_output=True, text=True)
print(validation.stdout)
assert validation.returncode == 0, validation.stderr
assert atlas_path.exists()
display(Image(filename=str(atlas_path)))

# Generate the same four-panel atlas from the real Tanager Tile-by-Tile outputs.
tanager_atlas_path = tanager_out/"tanager_advanced_atlas.png"
subprocess.run([
    sys.executable, str(atlas_tool), str(tanager_out), "--output", str(tanager_atlas_path),
    "--title", "Planet Tanager TRUST Tile-by-Tile",
    "--risk-name", "trust_anomaly_score.tif",
    "--priority-name", "trust_priority_classes.tif",
    "--stats-name", "TANAGER_HDF5_METADATA.json",
    "--manifest-name", "manifest.json"
], check=True)
assert tanager_atlas_path.exists()
print("Tanager atlas:", tanager_atlas_path)
display(Image(filename=str(tanager_atlas_path)))


## 9. Results interpretation
The application sample and Tanager track remain distinguishable but are executed in this one integrated notebook. The displayed zones are scene-relative candidates for human inspection. They are not confirmed disease, pest, nutrient, crop-stress, or other agricultural diagnoses. Rasterize-back metrics validate raster-to-vector geometry only; they are not agricultural ground-truth accuracy.


In [ ]:
# Results card for the application/frontend integration contract
result_card = {
    "track": "EnMAP Live Matrix sample demo",
    "scene": stats.get("scene"),
    "location": stats.get("location"),
    "crs": stats.get("crs"),
    "priority_zone_count": int(len(zones)),
    "artifacts": [str(p.relative_to(ROOT)) for p in [SAMPLE_RUN/"risk_map.tif", SAMPLE_RUN/"priority_map.tif", SAMPLE_RUN/"zones.csv", SAMPLE_RUN/"zones.geojson"]],
    "recommendation_label": "INSPECT / MONITOR / ABSTAIN are prioritization labels only",
    "caveat": "Field inspection and independent agronomic evidence are required before intervention.",
}
print(json.dumps(result_card, indent=2))
(Path(OUT/"result_card.json")).write_text(json.dumps(result_card, indent=2))

## 10. Reproducibility and provenance

The notebook uses fixed seeds, relative paths, explicit configuration, artifact validation, and no embedded secrets. Raw scenes are not written to Google Drive by default. For a submitted Colab link, set sharing to **Anyone with the link can view** and verify the link from a private window.

In [ ]:
# Provenance manifest for this notebook run
provenance = {
    "project_repo": PROJECT_REPO,
    "course_repo": COURSE_REPO,
    "seed": SEED,
    "scene_ids_recorded": SCENE_IDS,
    "tanager_status": TANAGER_STATUS,
    "tanager_attribution": "Planet Labs PBC, Tanager Core Imagery, CC-BY-4.0 when used",
    "raw_scene_policy": "Scene IDs only by default; no raw HDF5 persisted in Drive",
    "package_path": str(INTEGRATION.relative_to(ROOT)),
}
(Path(OUT/"provenance.json")).write_text(json.dumps(provenance, indent=2, default=str))
print(json.dumps(provenance, indent=2))

## 11. AI log and human verification
AI tools may draft notebook structure and code, but the team must verify every claim. This implementation was checked against Notebook 16 for static STAC indexing, scene filters, thumbnails/cloud-cover caution, and against Notebook 12 for lazy HDF5 reading, masks, good-band handling, window subsets, parameterized workflows, provenance, and memory-aware processing.

**Verified in this run:** the supplied real HDF5 was processed with Tile-by-Tile passes; the four-panel Tanager atlas was generated from the resulting GeoTIFFs; CRS, transform, dimensions, and geometry-integrity metrics were preserved.

**Still required for scientific validation:** an independent second scene and a synchronized cross-sensor reference. Both remain BLOCKED, and no agricultural accuracy claim is made.


## 12. Limitations and next steps
- The supplied real HDF5 has been processed into a separate Tanager output track inside the same integrated Colab workflow.
- The existing application artifacts are EnMAP-based; Tanager is a scientifically separate sensor track connected through shared provenance and frontend artifact contracts.
- Tile-by-Tile processing is enabled by default with `TILE_SIZE=128`; change it only after considering available Colab memory.
- Independent-scene validation and cross-sensor consistency remain BLOCKED because no second scene or compatible cross-sensor reference was supplied.
- TRUST and all thresholds are scene-relative prioritization signals, not calibrated uncertainty or agricultural ground truth.


## Final acceptance check
This notebook is the single integrated Google Colab entry point. It uses Scene IDs for discovery, reads the supplied HDF5 lazily by tiles, writes derived artifacts only, generates the matching four-panel atlas, and records provenance. The remaining BLOCKED items are independent-scene validation and cross-sensor consistency—not a missing Tanager execution.


In [ ]:
# Final machine-checkable status
checks = {
    "project_sample_artifacts": all((SAMPLE_RUN/n).exists() for n in required),
    "integration_package_present": INTEGRATION.exists(),
    "sample_atlas_generated": atlas_path.exists(),
    "sample_atlas_validation": validation.returncode == 0,
    "tanager_real_hdf5_available": TANAGER_HDF5_PATH.exists(),
    "tanager_tile_atlas_generated": (tanager_out/"tanager_advanced_atlas.png").exists(),
    "tanager_tile_by_tile": meta.get("method", "").startswith("Tile-by-Tile"),
    "independent_scene_validation": "BLOCKED",
    "cross_sensor_consistency": "BLOCKED",
}
print(json.dumps(checks, indent=2))
assert checks["project_sample_artifacts"] and checks["integration_package_present"]
assert checks["sample_atlas_generated"] and checks["sample_atlas_validation"]
assert checks["tanager_real_hdf5_available"] and checks["tanager_tile_atlas_generated"] and checks["tanager_tile_by_tile"]
print("Integrated Colab workflow: PASS")
print("Planet Tanager Tile-by-Tile experiment and atlas: PASS for supplied scene")
print("Independent-scene validation and cross-sensor consistency: BLOCKED")
